In [ ]:
# ── CELL 1: INSTALLATION ──────────────────────────
# Estimated time: 2-3 min. Wait for "DONE".

import os, shutil, subprocess, sys

# 1. Install system dependencies
!apt-get install -y zstd curl lsof -q 2>/dev/null

# 2. Install Ollama
OLLAMA_PATHS = [
    shutil.which("ollama"),
    "/usr/local/bin/ollama",
    "/usr/bin/ollama",
    os.path.expanduser("~/.local/bin/ollama"),
    os.path.expanduser("~/ollama"),
]
if not any(p and os.path.exists(p) for p in OLLAMA_PATHS):
    !curl -fsSL https://ollama.com/install.sh | sh
else:
    print("  Ollama already installed")

# 3. Install Open WebUI
try:
    import open_webui
    print("  Open WebUI already installed")
except ImportError:
    !pip install open-webui -q

# 4. Install pyngrok
try:
    import pyngrok
    print("  pyngrok already installed")
except ImportError:
    !pip install pyngrok -q

# 5. Verify all dependencies
missing = []
if not shutil.which("ollama"):
    ollama_found = any(p and os.path.exists(p) for p in OLLAMA_PATHS)
    if not ollama_found:
        missing.append("ollama")
try:
    import pyngrok
except ImportError:
    missing.append("pyngrok")
try:
    import open_webui
except ImportError:
    missing.append("open-webui")

if missing:
    print(f"FAILED: {', '.join(missing)} not found after installation")
else:
    print("=============================================")
    print("DONE - All dependencies installed. Proceed to Cell 2.")
    print("=============================================")


In [ ]:
# ── CELL 2: SETUP & RUN OLLAMA ─────────────────────
# Change model below if needed:
MODEL = "qwen2.5-coder:7b"   # default (~4.7GB)
# MODEL = "qwen2.5-coder:14b"  # coding (~9.0GB)
# MODEL = "deepseek-r1:7b"     # reasoning (~4.7GB)
# MODEL = "llama3.1:8b"        # general chat (~4.9GB)
# MODEL = "gemma4:12b"         # reasoning & coding (~6.6GB)

import subprocess, time, os, shutil, json, urllib.request, sys
from google.colab import drive

# --- Find ollama binary ---
OLLAMA_CANDIDATES = [
    shutil.which("ollama"),
    "/usr/local/bin/ollama",
    "/usr/bin/ollama",
    os.path.expanduser("~/.local/bin/ollama"),
    os.path.expanduser("~/ollama"),
]
ollama_bin = next((p for p in OLLAMA_CANDIDATES if p and os.path.exists(p)), None)
if not ollama_bin:
    print("Ollama not found - attempting install...")
    ret = os.system("curl -fsSL https://ollama.com/install.sh | sh")
    if ret != 0:
        print("Install failed. Run Cell 1 first.")
        sys.exit(1)
    ollama_bin = next((p for p in OLLAMA_CANDIDATES if p and os.path.exists(p)), None)
    if not ollama_bin:
        print("Binary not found after install.")
        sys.exit(1)
    print("  Ollama installed successfully")

# --- Mount Google Drive ---
print("[1/5] Mounting Google Drive...", flush=True)
if not os.path.exists("/content/drive"):
    drive.mount('/content/drive')
else:
    print("  Drive already mounted")

# --- Set up model paths ---
print("[2/5] Setting up model folder...")
DRIVE_PATH = "/content/drive/MyDrive/ollama_models"
LOCAL_PATH = os.path.expanduser("~/.ollama/models")

RESTORED_FLAG = os.path.join(LOCAL_PATH, ".restored")
if os.path.exists(DRIVE_PATH) and os.listdir(DRIVE_PATH):
    os.makedirs(LOCAL_PATH, exist_ok=True)
    if not os.path.exists(RESTORED_FLAG):
        print("  Restoring model from Drive...")
        for item in os.listdir(DRIVE_PATH):
            src = os.path.join(DRIVE_PATH, item)
            dst = os.path.join(LOCAL_PATH, item)
            if not os.path.exists(dst):
                try:
                    if os.path.isdir(src):
                        shutil.copytree(src, dst, dirs_exist_ok=True)
                    else:
                        shutil.copy2(src, dst)
                except Exception as e:
                    print(f"    Failed to copy {item}: {e}")
        open(RESTORED_FLAG, 'w').close()
        print("  Model restored")
    else:
        print("  Model already restored")
elif not os.path.exists(LOCAL_PATH):
    os.makedirs(LOCAL_PATH, exist_ok=True)

os.environ["OLLAMA_MODELS"] = LOCAL_PATH

# --- Kill stale Ollama processes ---
print("[3/5] Cleaning up stale processes...")
os.system("pkill -f 'ollama serve' 2>/dev/null || true")
time.sleep(1)

# --- Start Ollama ---
print("[4/5] Starting Ollama & loading model...")
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
proc_ollama = subprocess.Popen(
    [ollama_bin, "serve"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

# Wait for Ollama to be ready
print("  Waiting for Ollama...", end="", flush=True)
ollama_ready = False
for _ in range(30):
    try:
        r = urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        r.close()
        print(" OK")
        ollama_ready = True
        break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(2)
if not ollama_ready:
    print(" Ollama not responding")
    sys.exit(1)

# Pull model
print(f"  Pulling model {MODEL}...")
pull = subprocess.Popen(
    [ollama_bin, "pull", MODEL],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in iter(pull.stdout.readline, ''):
    line = line.rstrip()
    if line:
        print(f"  {line}")
pull.wait()

if pull.returncode != 0:
    print(f"  Pull failed (exit {pull.returncode})")
else:
    print("  Model pulled successfully")

# Backup model to Drive
if os.path.exists(LOCAL_PATH):
    print("  Backing up to Drive...")
    os.makedirs(DRIVE_PATH, exist_ok=True)
    for item in os.listdir(LOCAL_PATH):
        if item.startswith('.'):
            continue
        src = os.path.join(LOCAL_PATH, item)
        dst = os.path.join(DRIVE_PATH, item)
        if not os.path.exists(dst):
            try:
                if os.path.isdir(src):
                    shutil.copytree(src, dst, dirs_exist_ok=True)
                else:
                    shutil.copy2(src, dst)
            except Exception as e:
                print(f"    Backup failed for {item}: {e}")
    print("  Backup complete")

print("=============================================")
print(f"MODEL {MODEL} READY - Proceed to Cell 3.")
print("=============================================")


In [ ]:
# ── CELL 3: WEBUI + TUNNELS ──────────────────────────
# Cloudflare -> WebUI (port 8081) | ngrok -> Ollama (port 11434)
# MUST KEEP RUNNING while using WebUI / opencode

import os, subprocess, time, urllib.request, re, shutil, sys

WEBUI_PORT = 8081
OLLAMA_PORT = 11434

# --- Helper: kill process on a port ---
def kill_port(port):
    methods = [
        f"fuser -k {port}/tcp 2>/dev/null",
        f"lsof -ti:{port} | xargs kill -9 2>/dev/null",
    ]
    for cmd in methods:
        ret = os.system(cmd)
        if os.WIFEXITED(ret) and os.WEXITSTATUS(ret) == 0:
            return True
    return False

# ============================================================
# 1. START OPEN WEBUI
# ============================================================
webui_bin = shutil.which("open-webui") or shutil.which("open_webui")
if not webui_bin:
    print("open-webui not found. Run Cell 1 first.", flush=True)
    raise SystemExit(1)

print(f"[1/5] Cleaning port {WEBUI_PORT}...", flush=True)
kill_port(WEBUI_PORT)
time.sleep(1)

print(f"[2/5] Starting Open WebUI on port {WEBUI_PORT}...", flush=True)
env = {**os.environ, "OLLAMA_BASE_URL": "http://127.0.0.1:11434", "WEBUI_AUTH": "False"}
proc_webui = subprocess.Popen(
    [webui_bin, "serve", "--port", str(WEBUI_PORT)],
    env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

print("[3/5] Waiting for WebUI (1-3 min)...", flush=True)
for i in range(60):
    if proc_webui.poll() is not None:
        print(f"WebUI failed (exit {proc_webui.returncode})", flush=True)
        raise SystemExit(1)
    try:
        r = urllib.request.urlopen(f"http://127.0.0.1:{WEBUI_PORT}", timeout=5)
        r.close()
        print("WebUI is running!", flush=True)
        webui_ready = True
        break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(5)
else:
    print("WebUI timeout.", flush=True)

# ============================================================
# 2. SET UP TUNNELS
# ============================================================
print("[4/5] Setting up tunnels...", flush=True)

from pyngrok import ngrok

# --- Cloudflare -> WebUI (port 8081) ---
CLOUDFLARED_BIN = "/usr/local/bin/cloudflared"
if not os.path.exists(CLOUDFLARED_BIN):
    print("  Downloading cloudflared...", flush=True)
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        CLOUDFLARED_BIN
    )
    os.chmod(CLOUDFLARED_BIN, 0o755)

os.system("pkill -f cloudflared 2>/dev/null || true")
time.sleep(1)

print("  Cloudflare -> WebUI...", end="", flush=True)
proc_cf = subprocess.Popen(
    [CLOUDFLARED_BIN, "tunnel", "--url", f"http://127.0.0.1:{WEBUI_PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
webui_url = None
for _ in range(30):
    line = proc_cf.stdout.readline()
    if not line:
        break
    m = re.search(r'(https://[a-zA-Z0-9.-]+\.trycloudflare\.com)', line)
    if m:
        webui_url = m.group(1)
        print(" OK")
        break
    time.sleep(1)
    print(".", end="", flush=True)

# --- ngrok -> Ollama (port 11434) ---
# Get your free token at https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = ""  # <-- PASTE YOUR NGROK TOKEN HERE

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

print("  ngrok -> Ollama...", end="", flush=True)
try:
    tunnel = ngrok.connect(OLLAMA_PORT, "http")
    ollama_url = tunnel.public_url.replace("http://", "https://")
    print(" OK")
except Exception as e:
    print(f" ngrok failed: {e}")
    sys.exit(1)

# ============================================================
# 3. TEST TUNNELS
# ============================================================
print("[5/5] Testing tunnels...")

try:
    r = urllib.request.urlopen(f"{ollama_url}/api/tags", timeout=10)
    r.close()
    print("  Ollama accessible via ngrok")
except Exception as e:
    print(f"  Ollama check failed: {e}")

print()

# ============================================================
# 4. PRINT RESULTS
# ============================================================
SEP = "=" * 60
print(SEP)
print("WEBUI (open in browser):")
print(f"  {webui_url or 'FAILED'}")
print(SEP)
print()
print(SEP)
print("OLLAMA API (for opencode.json):")
print(f"  {ollama_url}")
print(SEP)
print()
print("Add this to opencode.json:")
print(SEP)
print("{")
print('  "$schema": "https://opencode.ai/config.json",')
print('  "provider": {')
print('    "Ollama": {')
print('      "npm": "@ai-sdk/openai-compatible",')
print('      "name": "Ollama (Colab)",')
print('      "options": {')
print(f'        "baseURL": "{ollama_url}/v1"')
print('      },')
print('      "models": {')
print(f'        "{MODEL}": {{')
print(f'          "name": "{MODEL}"')
print('        }')
print('      }')
print('    }')
print('  }')
print("}")
print()
print("NOTES:")
print("  - Keep this cell running while using opencode.")
print("  - If Colab disconnects, re-run Cell 1 -> 2 -> 3.")
print("  - Edit MODEL in Cell 2 to match your model.")
print(f"  - Model key in opencode.json is {MODEL} (must match API).")
print()

# ============================================================
# 5. MONITOR PROCESSES
# ============================================================
print("Monitoring services (Ctrl+C to stop)...")
try:
    while True:
        time.sleep(30)
        if proc_webui.poll() is not None:
            print("WebUI stopped!")
        if proc_cf.poll() is not None:
            print("Cloudflare tunnel stopped!")
            break
        if ngrok.get_ngrok_process().proc.poll() is not None:
            print("ngrok tunnel stopped!")
            break
        print("All active -", time.strftime("%H:%M:%S"))
except KeyboardInterrupt:
    print("Stopped.")
